In [ ]:
import os
import numpy as np


sample_dir = '/kaggle/input/datasets/vanshdistsys2/specifictask-9-a-dataset'

# 1. Check Directory Structure
print(f"Subdirectories found: {os.listdir(sample_dir)}")

# 2. Find one .npy file to probe
probe_file = None
for root, dirs, files in os.walk(sample_dir):
    for f in files:
        if f.endswith('.npy'):
            probe_file = os.path.join(root, f)
            break
    if probe_file: break

# 3. Probe the file geometry and stats
if probe_file:
    print(f"\nProbing file: {probe_file}")
    data = np.load(probe_file, allow_pickle=True)
    print(f"Raw dtype: {data.dtype}")
    
    # Unpack if necessary
    if data.dtype == object:
        data = data[0]
        
    print(f"Array Shape: {data.shape}")
    print(f"Data Type: {data.dtype}")
    print(f"Min Value: {data.min():.4f}")
    print(f"Max Value: {data.max():.4f}")
else:
    print("No .npy files found in the specified path.")

In [ ]:
"""
DeepLense Specific Test IX.A: Foundation Model for Gravitational Lensing
Author: Vansh Jain
Organization: ML4SCI

Master initialization cell. Consolidates all library dependencies, configures 
production logging streams, and enforces deterministic hardware execution.
"""

import os
import glob
import logging
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve, auc
from sklearn.preprocessing import label_binarize

# Configure production logging framework
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

# Enforce deterministic execution for reproducible baselines
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Compute device bound to: {device}")

In [ ]:
"""
Data Pipeline Configuration

Handles I/O, dynamic per-instance radiometric normalization, and strict 
cross-validation splitting of simulated lensing flux arrays. Isolates 
unperturbed Einstein rings for self-supervised pre-training.
"""

class DeepLenseFoundationDataset(Dataset):
    """
    Physically-constrained Dataset for parsing Task IX flux arrays.
    Enforces spatial dimensionality [1, 64, 64] and min-max normalization.
    """
    def __init__(self, root_dir):
        self.file_paths = glob.glob(os.path.join(root_dir, '**/*.npy'), recursive=True)
        
        if not self.file_paths:
            logging.error(f"No .npy arrays detected in {root_dir}.")
            raise FileNotFoundError(f"No .npy arrays detected in {root_dir}.")
            
        self.classes = sorted(list(set([os.path.basename(os.path.dirname(p)) for p in self.file_paths])))
        self.class_to_idx = {cls_name: idx for idx, cls_name in enumerate(self.classes)}
        
        logging.info(f"Topological classes mapped: {self.class_to_idx}")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        img_path = self.file_paths[idx]
        image_array = np.load(img_path, allow_pickle=True) 
        
        # Unpack 0-d object wrapper from simulation pipeline
        if image_array.dtype == object:
            image_array = image_array.item() if image_array.size == 1 else image_array[0]
            
        # Cast to float32 for PyTorch precision constraints
        image_array = np.array(image_array, dtype=np.float32)
        
        # Enforce strict Min-Max Normalization to [0, 1] to stabilize attention gradients
        arr_min = image_array.min()
        arr_max = image_array.max()
        if arr_max > arr_min:
            image_array = (image_array - arr_min) / (arr_max - arr_min + 1e-8)
            
        image_tensor = torch.from_numpy(image_array)
         
        # Enforce channel dimensionality [C, H, W]
        if len(image_tensor.shape) == 2:
            image_tensor = image_tensor.unsqueeze(0)
            
        parent_folder = os.path.basename(os.path.dirname(img_path))
        label = self.class_to_idx[parent_folder]
        
        return image_tensor, torch.tensor(label, dtype=torch.long)


dataset_path = '/kaggle/input/datasets/vanshdistsys2/specifictask-9-a-dataset' 
full_dataset = DeepLenseFoundationDataset(root_dir=dataset_path)

# Enforce strict 90:10 validation holdout to prevent representational leakage
total_size = len(full_dataset)
train_size = int(0.9 * total_size)
val_size = total_size - train_size

train_dataset, val_dataset = random_split(
    full_dataset, 
    [train_size, val_size], 
    generator=torch.Generator().manual_seed(42)
)

# Isolate baseline lenses ('no_sub') for MAE pre-training geometry
no_sub_idx = full_dataset.class_to_idx['no_sub']

# Efficiently filter training indices via directory paths to avoid I/O bottlenecks
no_sub_train_indices = [
    idx for idx in train_dataset.indices 
    if full_dataset.class_to_idx[os.path.basename(os.path.dirname(full_dataset.file_paths[idx]))] == no_sub_idx
]

pretrain_dataset = Subset(full_dataset, no_sub_train_indices)

pretrain_loader = DataLoader(pretrain_dataset, batch_size=32, shuffle=True, num_workers=2)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

logging.info(f"Total simulated arrays: {total_size}")
logging.info(f"MAE Pre-training subset (no_sub): {len(pretrain_dataset)}")
logging.info(f"Classification Train split: {len(train_dataset)}")
logging.info(f"Validation Holdout split: {len(val_dataset)}")

In [ ]:
"""
Foundation Model Architecture Specification

Implements a Vision Transformer (ViT) based Masked Autoencoder (MAE) optimized 
for extracting scale-invariant morphological features from simulated gravitational 
lensing flux arrays, followed by a linear classification head.
"""

# 1. The Masking Function----------------
def random_masking(x, mask_ratio=0.75):
    """
    Performs stochastic patch masking on the input sequence.
    Retains a subset of patches to force the encoder to learn global topological priors.
    """
    Batch, Seq_Len, Embed_Dim = x.shape
    len_keep = int(Seq_Len * (1 - mask_ratio))
    
    noise = torch.rand(Batch, Seq_Len, device=x.device) 
    ids_shuffle = torch.argsort(noise, dim=1) 
    ids_restore = torch.argsort(ids_shuffle, dim=1)
    
    ids_keep = ids_shuffle[:, :len_keep] 
    x_visible = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, Embed_Dim))
    
    # Generate binary mask matrix for isolated MSE loss calculation
    mask = torch.ones([Batch, Seq_Len], device=x.device)
    mask[:, :len_keep] = 0
    mask = torch.gather(mask, dim=1, index=ids_restore)
    
    return x_visible, mask, ids_restore
    
 #------ The Encoder----------

class MAE_Encoder(nn.Module):
    """
    ViT Encoder for mapping visible radiometric patches to latent space.
    """
    def __init__(self, image_size=64, patch_size=8, in_channels=1, embed_dim=256, num_heads=8):
        super().__init__()
        self.patch_embed = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (image_size // patch_size) ** 2 
        
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=512, activation="gelu", batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

    def forward(self, x, mask_ratio=0.75):
        batch_size = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        
        x_visible, mask, ids_restore = random_masking(x, mask_ratio) 
        
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x_visible = torch.cat((cls_tokens, x_visible), dim=1) 
        
        encoded_features = self.transformer(x_visible)
        return encoded_features, mask, ids_restore

"""
 The Decoder
"""

class MAE_Decoder(nn.Module):
    def __init__(self, image_size=64, patch_size=8, embed_dim=256, decoder_embed_dim=128, decoder_num_heads=4):
        super().__init__()
        num_patches = (image_size // patch_size) ** 2 
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, decoder_embed_dim))
        decoder_layer = nn.TransformerEncoderLayer(d_model=decoder_embed_dim, nhead=decoder_num_heads, dim_feedforward=256, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(decoder_layer, num_layers=2)
        self.pred = nn.Linear(decoder_embed_dim, patch_size ** 2)

    def forward(self, x, ids_restore):
        batch_size = x.shape[0]
        x = self.decoder_embed(x) 
        cls_token, x_visible = x[:, :1, :], x[:, 1:, :]   
        mask_tokens = self.mask_token.expand(batch_size, ids_restore.shape[1] - x_visible.shape[1], -1)
        x_full = torch.cat([x_visible, mask_tokens], dim=1) 
        x_full = torch.gather(x_full, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x_full.shape[2]))
        x_full = torch.cat([cls_token, x_full], dim=1) 
        x_full = x_full + self.decoder_pos_embed
        x_decoded = self.transformer(x_full)[:, 1:, :] 
        return self.pred(x_decoded)

#  The Master Autoencoder
class DeepLenseMAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = MAE_Encoder()
        self.decoder = MAE_Decoder()

    def forward(self, x, mask_ratio=0.75):
        encoded_features, mask, ids_restore = self.encoder(x, mask_ratio)
        predictions = self.decoder(encoded_features, ids_restore)
        return predictions, mask
        

#  The Fine-Tuning Classifier
class DeepLenseClassifier(nn.Module):
    def __init__(self, pretrained_encoder, embed_dim=256, num_classes=3):
        super().__init__()
        
        #  load pre-trained encoder
        self.encoder = pretrained_encoder
        
        # The final Classification Head
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(p=0.3),  
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, x):
        
        encoded_features, _, _ = self.encoder(x, mask_ratio=0.0)
        cls_token = encoded_features[:, 0, :] 
        logits = self.head(cls_token)
        
        return logits

In [ ]:
"""
Two-Stage Foundation Model Optimization & Telemetry

Phase 1: Self-supervised reconstruction of unperturbed topological priors.
Phase 2: Supervised classification of localized dark matter perturbations.
Integrates ReduceLROnPlateau scheduling and Early Stopping convergence criteria.
"""

PRETRAIN_EPOCHS = 10  # Increased ceiling; ES will terminate if optimal early
FINETUNE_EPOCHS = 20  # Increased ceiling; ES will terminate if optimal early

model = DeepLenseMAE().to(device)
optimizer_pre = optim.AdamW(model.parameters(), lr=1e-4)
patchifier = nn.Unfold(kernel_size=8, stride=8)

# ==============================================================================
# PHASE 1: Morphological Pre-Training (MAE)
# ==============================================================================
logging.info("Initiating Phase 1: Self-Supervised Pre-Training (Baseline Morphology)")
model.train()

for epoch in range(PRETRAIN_EPOCHS):
    loop = tqdm(pretrain_loader, desc=f"MAE Pre-Train Ep {epoch+1}/{PRETRAIN_EPOCHS}", leave=False)
    epoch_mse_loss = 0.0
    
    for images, _ in loop:
        images = images.to(device)
        targets = patchifier(images).transpose(1, 2).to(device)
        
        predictions, mask = model(images, mask_ratio=0.75)
        
        loss = (predictions - targets) ** 2
        loss = loss.mean(dim=-1)
        loss = (loss * mask).sum() / mask.sum()
        
        optimizer_pre.zero_grad()
        loss.backward()
        optimizer_pre.step()
        
        epoch_mse_loss += loss.item() * images.size(0)
        loop.set_postfix(MSE=loss.item())
        
    avg_train_loss = epoch_mse_loss / len(pretrain_loader.dataset)
    logging.info(f"MAE Pre-Train Epoch [{epoch+1}/{PRETRAIN_EPOCHS}] - Avg MSE Reconstruction Loss: {avg_train_loss:.6f}")

torch.save(model.state_dict(), 'deepLense_foundation_mae.pth')
logging.info("Phase 1 Complete. Foundation weights serialized.")


# ==============================================================================
# PHASE 2: Supervised Fine-Tuning with Validation Telemetry & Early Stopping
# ==============================================================================
logging.info("Initiating Phase 2: Supervised Fine-Tuning (Substructure Classification)")
classifier = DeepLenseClassifier(pretrained_encoder=model.encoder).to(device)

# Base optimizer and dynamic scheduler
optimizer_ft = optim.AdamW(classifier.parameters(), lr=5e-5, weight_decay=1e-4)
scheduler_ft = optim.lr_scheduler.ReduceLROnPlateau(optimizer_ft, mode='min', factor=0.5, patience=3)
criterion = nn.CrossEntropyLoss()

# Early Stopping parameters
best_val_loss = float('inf')
early_stop_patience = 5
epochs_without_improvement = 0

for epoch in range(FINETUNE_EPOCHS):
    # --- Training Pass ---
    classifier.train()
    epoch_train_loss = 0.0
    
    loop = tqdm(train_loader, desc=f"Classifier Fine-Tune Ep {epoch+1}/{FINETUNE_EPOCHS}", leave=False)
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)
        
        logits = classifier(images)
        loss = criterion(logits, labels)
        
        optimizer_ft.zero_grad()
        loss.backward()
        optimizer_ft.step()
        
        epoch_train_loss += loss.item() * images.size(0)
        loop.set_postfix(Train_CE=loss.item())
        
    avg_train_loss = epoch_train_loss / len(train_loader.dataset)
    
    # --- Validation Pass ---
    classifier.eval()
    epoch_val_loss = 0.0
    correct_preds = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            logits = classifier(images)
            loss = criterion(logits, labels)
            
            epoch_val_loss += loss.item() * images.size(0)
            preds = torch.argmax(logits, dim=1)
            correct_preds += (preds == labels).sum().item()
            
    avg_val_loss = epoch_val_loss / len(val_loader.dataset)
    val_accuracy = correct_preds / len(val_loader.dataset)
    
    # Broadcast current learning rate
    current_lr = optimizer_ft.param_groups[0]['lr']
    logging.info(f"Epoch [{epoch+1}/{FINETUNE_EPOCHS}] | LR: {current_lr:.2e} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy*100:.2f}%")
    
    # Step the learning rate scheduler based on validation plateau
    scheduler_ft.step(avg_val_loss)

    # --- Early Stopping & Checkpointing Logic ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
        torch.save(classifier.state_dict(), 'deepLense_classifier_best.pth')
        logging.info("Validation loss decreased. Model checkpoint updated.")
    else:
        epochs_without_improvement += 1
        logging.info(f"No improvement in validation loss for {epochs_without_improvement} epoch(s).")
        
        if epochs_without_improvement >= early_stop_patience:
            logging.info(f"Early Stopping triggered at epoch {epoch+1}. Restoring optimal weights.")
            break

# Restore the mathematically optimal weights before moving to Phase 3 Evaluation
classifier.load_state_dict(torch.load('deepLense_classifier_best.pth'))
logging.info("Phase 2 Complete. Champion model loaded into memory for final inference.")

In [ ]:
"""
Validation and Topological Metric Extraction

Evaluates the fine-tuned representations strictly against an isolated holdout set
to quantify classification performance on anomalous lensing signatures.
"""

logging.info("Initiating Phase 3: Validation Inference & Metrics Generation")

classifier.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating Holdout Set"):
        images = images.to(device)
        logits = classifier(images)
        
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

acc = accuracy_score(all_labels, all_preds)
auc_score = roc_auc_score(all_labels, all_probs, multi_class='ovr')

logging.info(f"Holdout Accuracy: {acc * 100:.2f}%")
logging.info(f"Multi-class ROC-AUC: {auc_score:.4f}")

classes = [0, 1, 2]
y_val_bin = label_binarize(all_labels, classes=classes)
y_score = np.array(all_probs)

plt.figure(figsize=(10, 8))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
class_names = [full_dataset.classes[i] for i in range(3)]

for i in range(3):
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2.5, label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Baseline')
plt.xlim([-0.02, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Foundation Model Validation: Substructure ROC-AUC')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig('TaskIX_ROC_Evaluation.png', dpi=300, bbox_inches='tight')

logging.info("Evaluation Complete. ROC topology artifacts saved.")
plt.show()

In [ ]:

#  Advanced Statistical Evaluation

from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

logging.info("Initiating Final Statistical Audit...")

# 1. Detailed Physics Report
report = classification_report(all_labels, all_preds, target_names=full_dataset.classes)
f1 = f1_score(all_labels, all_preds, average='weighted')

print("\n" + "="*60)
print(f"DEEPLENSE ViT-MAE PHYSICS REPORT | Weighted F1: {f1:.4f}")
print("="*60)
print(report)

# 2. Confusion Matrix Heatmap
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='magma', 
            xticklabels=full_dataset.classes, 
            yticklabels=full_dataset.classes)

plt.title('Substructure Confusion Matrix: True vs Predicted Physics', fontsize=14)
plt.ylabel('Ground Truth (Simulation)', fontsize=12)
plt.xlabel('Model Prediction', fontsize=12)
plt.tight_layout()
plt.savefig('TaskIX_Confusion_Matrix.png', dpi=300)
plt.show()

In [ ]:

#  Computational Complexity Audit


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(classifier)
encoder_params = count_parameters(classifier.encoder)
head_params = count_parameters(classifier.head)

print("\n" + "="*50)
print("VIT-MAE HARDWARE FOOTPRINT AUDIT")
print("="*50)
print(f"Total Trainable Parameters:   {total_params:,}")
print(f"Encoder (The Brain):          {encoder_params:,} ({encoder_params/total_params:.1%} of total)")
print(f"Classification Head:          {head_params:,}")
print("-" * 50)

# Calculate model size in Megabytes (assuming 32-bit floats)
model_size_mb = (total_params * 4) / (1024 ** 2)
print(f"Total Memory Footprint:       {model_size_mb:.2f} MB")
print("="*50)

In [ ]:

# SAVE THE BRAIN (Foundation Weights)

# We save the encoder specifically so we can load it into the Super-Res model later
encoder_path = 'deeplense_vit_encoder_brain.pth'
torch.save(classifier.encoder.state_dict(), encoder_path)

logging.info(f"Foundation 'Brain' successfully extracted and saved to {encoder_path}")